# Encoding Technique 2: One-Hot Encoding

**Dataset:** `Loan_Default.csv`

**When to use:** For **nominal** (unordered) categorical features with a **manageable number of unique categories**.

**Key concept:** For a feature with N unique categories, OHE creates **N new binary columns** — one per category. A `1` means that category is present; `0` means it's absent.

**Trade-off:** Increases dimensionality significantly. Can cause multicollinearity (see Dummy Encoding as the fix).

This notebook demonstrates two approaches:
- `category_encoders.OneHotEncoder` (more flexible)
- `sklearn.preprocessing.OneHotEncoder` (standard scikit-learn approach)

---

### Step 1: Setup, Data Loading & Prep

In [ ]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
import category_encoders as ce

# Load data from the raw directory
df = pd.read_csv('../../data/raw/Loan_Default.csv')
df.drop(['ID', 'year'], axis=1, inplace=True)

categorical_features = df.select_dtypes(include=['object']).columns.tolist()
Ordinal_features = ['age']
Nominal_features = categorical_features.copy()
Nominal_features.remove('age')

# Pre-encode the ordinal feature so the baseline data is clean
enc = OrdinalEncoder()
df[Ordinal_features] = enc.fit_transform(df[Ordinal_features])

print(f'Shape before OHE: {df.shape}')
df.head()

---
## Approach A: `category_encoders.OneHotEncoder`

In [ ]:
df_onehot_ce = df.copy()

# Configure the encoder
OH_encoder = ce.OneHotEncoder(
    cols=Nominal_features,
    handle_unknown='return_nan',
    return_df=True,
    use_cat_names=True   # gives descriptive names like 'Gender_Male'
)

In [ ]:
# Separate and encode
df_onehot_ce_numerical = df_onehot_ce.drop(Nominal_features, axis=1)
df_onehot_ce_categorical = OH_encoder.fit_transform(df_onehot_ce[Nominal_features])

# Reassemble
df_onehot_ce = pd.concat([df_onehot_ce_numerical, df_onehot_ce_categorical], axis=1)

print(f'Shape after OHE (category_encoders): {df_onehot_ce.shape}')
df_onehot_ce.columns.tolist()

---
## Approach B: `sklearn.preprocessing.OneHotEncoder`

In [ ]:
df_onehot_sk = df.copy()

OH_encoder_sk = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
df_onehot_sk_categorical = pd.DataFrame(OH_encoder_sk.fit_transform(df_onehot_sk[Nominal_features]))

# Assign descriptive column names
df_onehot_sk_categorical.columns = OH_encoder_sk.get_feature_names_out(Nominal_features)
df_onehot_sk_categorical.index = df_onehot_sk.index

# Remove original categorical columns and add encoded ones
df_onehot_sk_numerical = df_onehot_sk.drop(Nominal_features, axis=1)
df_onehot_sk = pd.concat([df_onehot_sk_numerical, df_onehot_sk_categorical], axis=1)

print(f'Shape after OHE (sklearn): {df_onehot_sk.shape}')
df_onehot_sk.head()

### Key Observation

Notice how the number of columns **increased dramatically** after One-Hot Encoding. This is the "curse of dimensionality" trade-off. For the `Gender` column alone (4 categories), OHE created 4 new binary columns.